# 01d — Nền benign thêm: QoS (app tiêu dùng) và ZeroTier

Hai nguồn nền "khó" cho 03b, dùng giống `benign_tii` (01c): mỗi file tính cửa sổ riêng, không trộn với tấn công,
mọi host LAN gán `benign`. Mục đích là giảm báo nhầm trên traffic thật mà nền UNSW/TII chưa có
(xem `update_benign_qos_zerotier.md`): ZeroTier bị báo `recon`, gọi video/stream/tải file bị báo `dos`.

**QoS** (`Data/pcap_data_QoS`: gọi thoại/video, video ngắn/dài, game, tải file, chạy nền; PC/iOS/Android; mỗi app bắt 5 lần):
1. Đọc bằng tshark, **không dừng khi file lỗi**: file cụt giữa chừng giữ phần đọc được, file hỏng hoàn toàn (0 gói) bỏ.
2. **Tách gói gộp.** File dải `192.168.239.` được bắt ngay trên máy nên có gói TCP dài hơn MTU (card mạng gộp nhiều đoạn,
   GRO/TSO; tới 7,8% số gói ở Android). Router chỉ thấy gói ≤ 1514 B, nên các gói này được tách lại theo MSS.
3. **Gỡ NAT chiều về.** Pcap dải `192.168.11.` / `192.168.5.` bắt sau 2 lớp NAT: chiều đi có IP LAN thật của máy, còn
   chiều về gửi tới IP WAN của NAT (vd `192.168.138.15`). Không gỡ thì mỗi máy trông như đang flood mà không nhận được
   phản hồi nào. Gói từ ngoài vào được đổi `ip_dst` thành máy LAN đã gửi tới đúng (IP, port) phía xa đó; không khớp port
   thì theo IP phía xa. File dải `192.168.239.` không bị NAT chiều về, bước này không đổi gì.
4. **Đổi dải LAN** của file (`192.168.11.`, `192.168.239.`, `192.168.5.`) về `192.168.1.` như 03b.
   IP `192.168.1.x` có sẵn trong file (không thuộc LAN của file), nếu có, được chuyển ra `172.31.1.x`.
5. **Chia theo lần bắt** (số trong tên file): lần 1–3 vào train, lần 4 vào val, lần 5 vào test, để app nào cũng có ở cả 3 tập.

**ZeroTier** (`Data/pcap_zerotier/{train,val,test}/*.pcap`, tự bắt trên router): split chọn bằng thư mục, nên chia theo máy
hoặc theo trường hợp; `pc71.pcap` để ở `test/`. Giữ nguyên địa chỉ (đã là `192.168.1.x`).
Mỗi file train thêm `ZT_AUG` bản tăng cường, chỉ đổi gói ZeroTier: số peer, số port cục bộ, tỉ lệ phản hồi, nhịp đợt.
Không chỉ đổi IP/port, vì đặc trưng không dùng IP/port cụ thể nên đổi địa chỉ không tạo ra mẫu mới.
Khi chưa đủ pcap thật, train/val dùng thêm **ZeroTier mô phỏng** (mục 3b, `ZT_SIM`).

Đầu ra:
- `Data_after_processing/benign_qos/<file>.parquet`, `Data_after_processing/benign_zt/<split>_<file>[_augK].parquet` (cột như 01/01c)
- `reports/01d_extra_parts.csv`: mỗi đoạn một dòng (`source`, `name`, `split`, `scenario`, `sample_id`, `parquet`, …); 03b đọc file này
- `reports/01d_qos_files.csv`: tình trạng từng file QoS, kể cả file hỏng

In [1]:
import os, re, subprocess, sys, zlib
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd

ROOT = Path('/home/ubuntu/sepcung/03.Brute_Force_vs_Recon')
sys.path.insert(0, str(ROOT / 'Code' / 'deploy'))
import pkt_features as pf  # FIELDS/COLS như 01/01c; window_features (giống 03b) chỉ dùng ở mục kiểm tra

QOS_DIR = ROOT / 'Data' / 'pcap_data_QoS'
ZT_DIR = ROOT / 'Data' / 'pcap_zerotier'  # train/, val/, test/: pcap tự bắt trên router
OUT = ROOT / 'Data_after_processing'
QOS_OUT, ZT_OUT = OUT / 'benign_qos', OUT / 'benign_zt'
REPORT_DIR = OUT / 'reports'

LAN_PREFIX = '192.168.1.'     # khớp 03b
OUTSIDE_PREFIX = '172.31.1.'  # IP 192.168.1.x có sẵn trong pcap QoS (không thuộc LAN của file) chuyển ra đây
QOS_SPLIT = {1: 'train', 2: 'train', 3: 'train', 4: 'val', 5: 'test'}  # theo lần bắt (số trong tên file)
QOS_NAME = re.compile(r'^(?P<cat>[A-Za-z]+)_(?P<app>[A-Za-z0-9]+?)_?(?P<idx>\d+)_?(?P<device>[a-z]+)\.pcap$')
DEVICE = {'adr': 'android'}
MTU_FRAME = 1514         # gói Ethernet lớn nhất router thấy (MTU 1500 + 14 B header)
ZT_PORT = 9993
ZT_AUG = 5              # số bản tăng cường cho mỗi file ZeroTier train (0 = tắt)
ZT_BURST_GAP_S = 1.0     # gói ZeroTier cách gói trước > 1 s thì sang đợt mới
ZT_BURST_MAX_S = 5.0     # chỉ co giãn đợt ngắn (dò đường, hole punching), không đụng phiên truyền dữ liệu dài
ZT_SIM = {'train': 300, 'val': 60}  # số mẫu ZeroTier mô phỏng mỗi split (0 = tắt); test chỉ dùng pcap thật
ZT_SIM_BG = 0.4          # tỉ lệ mẫu mô phỏng ghép vào traffic của máy chính trong một file QoS cùng split
SEED = 42
N_JOBS = 12

for d in [QOS_OUT, ZT_OUT, *(ZT_DIR / s for s in ['train', 'val', 'test'])]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Đọc pcap chịu lỗi

In [2]:
PRIV = r'^(?:10\.|192\.168\.|172\.(?:1[6-9]|2\d|3[01])\.)'
MULTI = r'^(?:22[4-9]|23\d|255)\.'


def read_pcap_tolerant(pcap):
    """Như pf.read_pcap (cùng trường/cột với 01/01c) nhưng không dừng khi tshark báo lỗi: file cụt giữa chừng
    vẫn giữ các gói đọc được trước chỗ hỏng. Trả về (bảng gói sắp theo thời gian, tshark có báo lỗi không)."""
    cmd = ['tshark', '-r', str(pcap), '-n', '-T', 'fields', '-E', 'separator=\t', '-E', 'occurrence=f']
    for f in pf.FIELDS:
        cmd += ['-e', f]
    r = subprocess.run(cmd, capture_output=True, text=True, env=dict(os.environ, LC_ALL='C'))
    lines = [l.split('\t') for l in r.stdout.splitlines() if l]
    lines = [l for l in lines if len(l) == len(pf.COLS)]  # dòng cuối có thể bị cắt khi tshark dừng giữa chừng
    df = pd.DataFrame(lines, columns=pf.COLS)
    df['ts'] = pd.to_numeric(df['ts'], errors='coerce')
    for c in pf.INT_COLS:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int32')
    df['tcp_flags'] = df['tcp_flags'].map(lambda x: int(x, 16) if x else pd.NA).astype('Int32')
    # ip.proto lấy lớp ngoài: cổng TCP/UDP bên trong ICMP error không được tính là gói TCP/UDP
    is_icmp = df['ip_proto'] == 1
    df.loc[is_icmp, ['tcp_sport', 'tcp_dport', 'tcp_flags', 'tcp_len', 'udp_sport', 'udp_dport']] = pd.NA
    for c in ['eth_src', 'eth_dst', 'arp_sha']:
        df[c] = df[c].str.lower()
    df = df.dropna(subset=['ts']).sort_values('ts', kind='stable').reset_index(drop=True)
    df['label'] = ''
    return df, r.returncode != 0


def part_row(pk, path):
    """Thông tin chung của một đoạn cho reports/01d_extra_parts.csv (giống 01c_tii_benign_parts.csv)."""
    pair = pk['ip_src'].fillna('') + '>' + pk['ip_dst'].fillna('')
    return {'pkts': len(pk), 't0': pk['ts'].min(), 't1': pk['ts'].max(),
            'hours': (pk['ts'].max() - pk['ts'].min()) / 3600,
            'lan_hosts': pk.loc[pk['ip_src'].fillna('').str.startswith(LAN_PREFIX), 'ip_src'].nunique(),
            'top_pair': pair.value_counts().index[0], 'parquet': str(path)}

## 2. QoS: gỡ NAT, đổi dải LAN, chia theo lần bắt

In [3]:
def denat(pk, prefix):
    """Gói từ ngoài gửi tới IP WAN của NAT: ip_dst = máy LAN đã gửi tới đúng (IP, port) phía xa đó; không khớp port
    thì lấy máy gửi nhiều nhất tới IP phía xa đó. Trả về (số gói chiều về cần gỡ, số gói gỡ được)."""
    src, dst = pk['ip_src'].fillna(''), pk['ip_dst'].fillna('')
    lan_s, lan_d = src.str.startswith(prefix), dst.str.startswith(prefix)
    port = lambda a, b: pk[a].astype('float').fillna(pk[b].astype('float')).fillna(-1)
    inc = ~lan_s & ~lan_d & (src != '') & (dst != '') & ~dst.str.match(MULTI) & ~dst.str.endswith('.255')
    out = pd.DataFrame({'r': dst, 'p': port('tcp_dport', 'udp_dport'), 'dev': src})[lan_s & ~lan_d & (dst != '')]
    if out.empty or not inc.any():
        return int(inc.sum()), 0
    top = lambda s: s.value_counts().index[0]
    by_port, by_ip = out.groupby(['r', 'p'])['dev'].agg(top), out.groupby('r')['dev'].agg(top)
    key = pd.MultiIndex.from_arrays([src[inc], port('tcp_sport', 'udp_sport')[inc]])
    new = pd.Series(by_port.reindex(key).to_numpy(), index=src[inc].index)
    new = new.fillna(pd.Series(by_ip.reindex(src[inc]).to_numpy(), index=new.index)).dropna()
    pk.loc[new.index, 'ip_dst'] = new
    return int(inc.sum()), len(new)


def split_offload(pk):
    """Gói TCP dài hơn MTU: card mạng gộp nhiều đoạn thành một gói (GRO/TSO) vì pcap được bắt ngay trên máy, còn router
    thì thấy từng gói ≤ 1514 B. Tách lại theo MSS = 1514 − header (54 hoặc 66 B); các đoạn cách nhau 10 µs.
    Trả về (bảng gói, số gói gộp đã tách)."""
    big = ((pk['len'] > MTU_FRAME) & (pk['ip_proto'] == 6) & (pk['tcp_len'] > 0)).fillna(False).to_numpy()
    if not big.any():
        return pk, 0
    b = pk[big]
    hdr = (b['len'] - b['tcp_len']).astype(np.int64)
    mss, pay = MTU_FRAME - hdr, b['tcp_len'].astype(np.int64)
    rep = b.loc[b.index.repeat(np.ceil(pay / mss).astype(np.int64))].copy()
    k = rep.groupby(level=0).cumcount().to_numpy()
    seg = np.minimum(mss.loc[rep.index].to_numpy(), pay.loc[rep.index].to_numpy() - k * mss.loc[rep.index].to_numpy())
    rep['tcp_len'] = pd.array(seg, dtype='Int32')
    rep['len'] = pd.array(hdr.loc[rep.index].to_numpy() + seg, dtype='Int32')
    rep['ts'] = rep['ts'].to_numpy() + k * 1e-5
    return pd.concat([pk[~big], rep]).sort_values('ts', kind='stable', ignore_index=True), int(big.sum())


def swap_prefix(pk, old, new):
    """Đổi tiền tố IP old -> new ở mọi cột địa chỉ IP/ARP. Trả về số ô đã đổi."""
    n = 0
    for c in ['ip_src', 'ip_dst', 'arp_spa', 'arp_tpa']:
        s = pk[c].fillna('')
        m = s.str.startswith(old)
        pk.loc[m, c] = new + s[m].str.slice(len(old))
        n += int(m.sum())
    return n


def process_qos(fname):
    m = QOS_NAME.match(fname)
    name, idx = Path(fname).stem, int(m['idx'])
    row = {'source': 'benign_qos', 'name': name, 'sample_id': f'qos_{name}', 'split': QOS_SPLIT[idx],
           'scenario': f"qos_{m['cat'].lower()}", 'cat': m['cat'].lower(), 'app': m['app'].lower(), 'idx': idx,
           'device': DEVICE.get(m['device'], m['device'])}
    pk, broken = read_pcap_tolerant(QOS_DIR / fname)
    src = pk['ip_src'].fillna('')
    priv = src[src.str.match(PRIV)]
    if priv.empty:  # file hỏng hoàn toàn (0 gói) hoặc không có máy LAN
        return row | {'status': 'hong', 'pkts': len(pk)}
    dev = priv.value_counts().index[0]  # máy chính = IP private gửi nhiều gói nhất
    prefix = dev.rsplit('.', 1)[0] + '.'
    pk, n_split = split_offload(pk)
    n_in, n_map = denat(pk, prefix)
    moved = 0
    if prefix != LAN_PREFIX:
        moved = swap_prefix(pk, LAN_PREFIX, OUTSIDE_PREFIX)
        swap_prefix(pk, prefix, LAN_PREFIX)
    path = QOS_OUT / f'{name}.parquet'
    pk.to_parquet(path, index=False)
    return row | {'status': 'cut' if broken else 'ok', 'lan_prefix_orig': prefix,
                  'dev_ip': LAN_PREFIX + dev.rsplit('.', 1)[1], 'offload_split': n_split, 'inbound': n_in,
                  'inbound_mapped': n_map, 'ip_moved_out': moved, **part_row(pk, path)}


for p in QOS_OUT.glob('*.parquet'):  # chạy lại từ đầu, không để sót đoạn của lần trước
    p.unlink()
files = [p.name for p in sorted(QOS_DIR.glob('*.pcap'), key=lambda p: -p.stat().st_size)]  # file lớn chạy trước
bad = [f for f in files if not QOS_NAME.match(f)]
assert not bad, f'tên file QoS không đúng mẫu <nhóm>_<app>_<lần>_<thiết bị>.pcap: {bad[:5]}'
with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
    qos = pd.DataFrame(list(ex.map(process_qos, files))).sort_values('name', ignore_index=True)
ok = qos[qos['status'] != 'hong']
print(f"file: {len(qos)} | hỏng hoàn toàn (bỏ): {(qos['status'] == 'hong').sum()} | "
      f"cụt giữa chừng (giữ phần đọc được): {(qos['status'] == 'cut').sum()} | "
      f"gói gộp > MTU đã tách: {ok['offload_split'].sum():.0f} | "
      f"gói chiều về gỡ NAT được: {ok['inbound_mapped'].sum() / max(ok['inbound'].sum(), 1):.1%} | "
      f"ô IP 192.168.1.x có sẵn phải chuyển ra ngoài: {ok['ip_moved_out'].sum():.0f}")
pd.concat({'file': ok.pivot_table(index='cat', columns='split', values='name', aggfunc='size', fill_value=0),
           'giờ': ok.pivot_table(index='cat', columns='split', values='hours', aggfunc='sum', fill_value=0).round(2)},
          axis=1)

file: 430 | hỏng hoàn toàn (bỏ): 63 | cụt giữa chừng (giữ phần đọc được): 25 | gói gộp > MTU đã tách: 108470 | gói chiều về gỡ NAT được: 98.3% | ô IP 192.168.1.x có sẵn phải chuyển ra ngoài: 815


file             giờ            
split      test train val  test train   val
cat                                        
background    1     6   1  0.04  0.23  0.04
download      2     7   2  0.01  0.03  0.01
game         10    34  11  1.16  3.19  0.85
longvideo    14    42  14  0.39  1.15  0.40
shortvideo   15    44  15  0.32  0.72  0.30
videocall    14    45  15  0.46  1.43  0.50
voicecall    15    45  15  0.64  2.05  0.71

In [4]:
# Tỉ lệ gói chiều về gỡ NAT được theo dải LAN gốc và thiết bị
ok.groupby(['lan_prefix_orig', 'device']).agg(
    file=('name', 'size'), inbound=('inbound', 'sum'), mapped=('inbound_mapped', 'sum'),
    lan_hosts_max=('lan_hosts', 'max')).assign(mapped_share=lambda d: (d['mapped'] / d['inbound']).round(3))

file    inbound     mapped  lan_hosts_max  \
lan_prefix_orig device                                               
192.168.11.     android   100  2011913.0  1923065.0            3.0   
                ios       119  2461658.0  2393683.0            2.0   
                pc         80  5120871.0  5111792.0            3.0   
192.168.239.    android    19       77.0       77.0           23.0   
                ios         2        0.0        0.0           17.0   
                pc         46        0.0        0.0           27.0   
192.168.5.      ios         1    48435.0    48042.0            2.0   

                         mapped_share  
lan_prefix_orig device                 
192.168.11.     android         0.956  
                ios             0.972  
                pc              0.998  
192.168.239.    android         1.000  
                ios               NaN  
                pc                NaN  
192.168.5.      ios             0.992

## 3. ZeroTier: nhận diện gói, tăng cường cho train

Gói ZeroTier là gói UDP có port 9993, hoặc gói gửi từ/tới **port phụ**. Port phụ là port UDP mà host LAN đã dùng để gửi
tới port 9993: ZeroTier chọn ngẫu nhiên 2 port này lúc cài (vd `41940`, `51284` ở máy .71).

Mỗi bản tăng cường (seed riêng theo tên bản) chỉ đổi gói ZeroTier, traffic khác của máy giữ nguyên:
- **Số peer:** giữ ngẫu nhiên 40–100% IP phía xa. Sau đó nhân bản 0–100% số peer còn lại sang IP public mới:
  lệch thời gian 0–50 ms, port phía xa khác 9993 đổi ngẫu nhiên. Số đích trong mỗi cửa sổ vì vậy thay đổi.
- **Số port cục bộ** (30% số bản): bỏ mọi gói của một port phụ.
- **Phản hồi:** giữ 0% (25% số bản), 0–100% (50%) hoặc 100% (25%) gói ZeroTier chiều về.
- **Nhịp đợt:** trong các đợt ngắn (≤ `ZT_BURST_MAX_S`), khoảng cách giữa các gói được co giãn ×0,3–3 (log-uniform).

In [5]:
def _zt_cols(pk):
    src, dst = pk['ip_src'].fillna(''), pk['ip_dst'].fillna('')
    sp = pk['udp_sport'].astype('float').fillna(-1).astype(np.int64)
    dp = pk['udp_dport'].astype('float').fillna(-1).astype(np.int64)
    return src, dst, sp, dp, src + ':' + sp.astype(str), dst + ':' + dp.astype(str)


def zt_mask(pk):
    """Gói ZeroTier: UDP có port 9993, hoặc gửi từ/tới port phụ của host LAN (port UDP host đó đã dùng để gửi tới 9993).
    Trả về (mask, tập 'ip:port' cục bộ của ZeroTier)."""
    src, dst, sp, dp, ks, kd = _zt_cols(pk)
    local = set(ks[src.str.startswith(LAN_PREFIX) & (dp == ZT_PORT)])
    return ((sp == ZT_PORT) | (dp == ZT_PORT) | ks.isin(local) | kd.isin(local)).to_numpy(), local


def _rand_public(rng):
    return f'{rng.integers(11, 100)}.{rng.integers(0, 256)}.{rng.integers(0, 256)}.{rng.integers(1, 255)}'


def _map_ports(c, col, mask, pmap):
    v = c[col].astype('float')
    c[col] = v.where(~mask, v.map(lambda x: pmap.get(x, x))).astype('Int32')


def augment_zt(pk, rng):
    """Biến thể ZeroTier cho train: chỉ đổi gói ZeroTier, traffic khác giữ nguyên. Trả về (gói mới, mô tả)."""
    zt, local = zt_mask(pk)
    src, dst, sp, dp, ks, kd = _zt_cols(pk)
    out = src.str.startswith(LAN_PREFIX).to_numpy()  # gói do host LAN gửi
    remote = np.where(out, dst, src)
    desc = {}
    # số peer: giữ 40–100% IP phía xa
    peers = np.unique(remote[zt])
    dropped = peers[rng.random(len(peers)) > rng.uniform(0.4, 1.0)]
    keep = ~(zt & np.isin(remote, dropped))
    # số port cục bộ: bỏ một port phụ
    sec = sorted(k for k in local if not k.endswith(f':{ZT_PORT}'))
    desc['drop_port'] = bool(sec) and rng.random() < 0.3
    if desc['drop_port']:
        k = sec[rng.integers(len(sec))]
        keep &= ~((ks == k) | (kd == k)).to_numpy()
    # phản hồi chiều về
    u = rng.random()
    rk = 0.0 if u < 0.25 else (rng.uniform(0, 1) if u < 0.75 else 1.0)
    keep &= ~(zt & ~out & (rng.random(len(pk)) >= rk))
    desc['resp_keep'] = round(rk, 2)
    # thêm peer: nhân bản gói của một phần peer còn lại sang IP public mới
    z, z_remote = pk[keep & zt], remote[keep & zt]
    kept = np.unique(z_remote)
    n_new = int(round(len(kept) * rng.uniform(0, 1)))
    clones = []
    for p in (rng.choice(kept, n_new, replace=False) if n_new else []):
        c = z[z_remote == p].copy()
        ip = _rand_public(rng)
        o = c['ip_dst'] == p  # chiều đi
        c.loc[o, 'ip_dst'] = ip
        c.loc[~o, 'ip_src'] = ip
        rp = pd.concat([c.loc[o, 'udp_dport'], c.loc[~o, 'udp_sport']]).astype('float').dropna().unique()
        pmap = {x: float(rng.integers(1024, 65536)) for x in rp if x != ZT_PORT}  # peer sau NAT: port phía xa ngẫu nhiên
        _map_ports(c, 'udp_dport', o, pmap)
        _map_ports(c, 'udp_sport', ~o, pmap)
        c['ts'] = c['ts'] + rng.uniform(0, 0.05)
        clones.append(c)
    desc['peers'] = f'{len(peers)}->{len(kept) + n_new}'
    pk = pd.concat([pk[keep], *clones], ignore_index=True).sort_values('ts', kind='stable', ignore_index=True)
    # nhịp đợt: co giãn khoảng cách giữa các gói ZeroTier trong đợt ngắn
    f = float(np.exp(rng.uniform(np.log(0.3), np.log(3))))
    i = np.flatnonzero(zt_mask(pk)[0])
    if len(i):
        t = pk['ts'].to_numpy()[i]
        g = pd.Series(t).groupby(np.r_[0, np.cumsum(np.diff(t) > ZT_BURST_GAP_S)])
        t0, t1 = g.transform('min').to_numpy(), g.transform('max').to_numpy()
        short = (t1 - t0) <= ZT_BURST_MAX_S
        ts = pk['ts'].to_numpy().copy()
        ts[i[short]] = t0[short] + (t[short] - t0[short]) * f
        pk['ts'] = ts
    desc['time_scale'] = round(f, 2)
    return pk.sort_values('ts', kind='stable', ignore_index=True), desc


def process_zt(job):
    path, split = Path(job['path']), job['split']
    pk0, broken = read_pcap_tolerant(path)
    if pk0.empty:
        return [{'source': 'benign_zt', 'name': f'{split}_{path.stem}', 'split': split, 'status': 'hong', 'pkts': 0}]
    rows = []
    for k in range(ZT_AUG + 1 if split == 'train' else 1):  # bản gốc + ZT_AUG bản tăng cường (chỉ train)
        name = f'{split}_{path.stem}' + (f'_aug{k}' if k else '')
        pk, desc = (pk0, {}) if k == 0 else augment_zt(pk0, np.random.default_rng(zlib.crc32(f'{name}/{SEED}'.encode())))
        out = ZT_OUT / f'{name}.parquet'
        pk.to_parquet(out, index=False)
        rows.append({'source': 'benign_zt', 'name': name, 'sample_id': f'zt_{name}', 'split': split,
                     'scenario': 'zerotier', 'file': f'{split}/{path.name}', 'aug': k, 'status': 'cut' if broken else 'ok',
                     'zt_pkts': int(zt_mask(pk)[0].sum()), **part_row(pk, out),
                     **{f'aug_{a}': v for a, v in desc.items()}})
    return rows


for p in ZT_OUT.glob('*.parquet'):
    p.unlink()
zt_jobs = [{'path': str(p), 'split': s} for s in ['train', 'val', 'test'] for p in sorted((ZT_DIR / s).glob('*.pcap*'))]
with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
    zt = pd.DataFrame([r for rows in ex.map(process_zt, zt_jobs) for r in rows])
if zt.empty or not (zt['split'] == 'train').any():
    print('Chưa có pcap ZeroTier train (Data/pcap_zerotier/train/): benign_zt chỉ có ở val/test, model chưa học được ZeroTier.')
zt.drop(columns=['parquet', 't0', 't1'], errors='ignore').round(3)

Chưa có pcap ZeroTier train (Data/pcap_zerotier/train/): benign_zt chỉ có ở val/test, model chưa học được ZeroTier.


,source,name,sample_id,split,scenario,file,aug,status,zt_pkts,pkts,hours,lan_hosts,top_pair
0,benign_zt,test_pc71,zt_test_pc71,test,zerotier,test/pc71.pcap,0,ok,514,579,0.005,2,192.168.1.71>113.190.37.142


## 3b. ZeroTier mô phỏng (tới khi có đủ pcap ZeroTier để train)

Hiện chỉ có `pc71.pcap` (17 s), và file này dùng để test. Vì vậy train/val dùng traffic ZeroTier **mô phỏng theo cách
giao thức hoạt động**. Tham số rút ngẫu nhiên trong khoảng rộng, không chỉnh cho khớp `pc71.pcap`:
- **Keepalive** tới 3–5 root (IP public, port 9993) mỗi 20–60 s; mỗi mẫu có tỉ lệ trả lời 30–100%.
- **Peer:** 2–60 peer, mỗi peer có 1–6 IP ứng viên (IP public, IP private của LAN khác, docker `172.17.0.1`).
  Mỗi IP là một node: nghe 9993 (60%) và/hoặc 0–2 port riêng (port phụ, port do NAT ánh xạ).
- **Dò đường** (hole punching), trung bình mỗi 5–60 s một đợt ngắn: gửi 1–2 gói dò 70 B từ mọi port cục bộ (1–3 port)
  tới mọi địa chỉ ứng viên của một số peer (20% số đợt xen 10% gói HELLO 145–210 B). Các gói cách nhau 10 µs – 10 ms,
  có 0–20% địa chỉ trả lời.
- **Phiên dữ liệu** qua ZeroTier ở 30% số mẫu: UDP hai chiều 3–30 gói/s, kéo dài 10–120 s.

Một phần số mẫu (`ZT_SIM_BG` = 40%) được ghép vào traffic của máy chính trong một file QoS cùng split (cắt 60–180 s);
số còn lại là máy gần như rảnh, chỉ chạy ZeroTier (đúng trường hợp máy .71 bị báo nhầm). Khi đã có pcap thật trong `train/` thì nên giảm `ZT_SIM` hoặc tắt (đặt 0).

In [6]:
GW_MAC, HOST_MAC = '02:00:00:00:00:01', '02:00:00:00:00:02'


def _rand_private(rng):
    while True:
        k = rng.integers(4)
        ip = ('172.17.0.1' if k == 3 else
              f'192.168.{rng.integers(0, 256)}.{rng.integers(1, 255)}' if k == 0 else
              f'10.{rng.integers(0, 256)}.{rng.integers(0, 256)}.{rng.integers(1, 255)}' if k == 1 else
              f'172.{rng.integers(16, 32)}.{rng.integers(0, 256)}.{rng.integers(1, 255)}')
        if not ip.startswith(LAN_PREFIX):
            return ip


def zt_sim(host, t0, t1, rng):
    """Traffic ZeroTier mô phỏng của host trong [t0, t1]. Trả về bảng gói cùng cột/kiểu với read_pcap_tolerant."""
    local = [ZT_PORT] + [int(p) for p in rng.integers(20000, 65536, rng.choice([0, 1, 2], p=[0.2, 0.3, 0.5]))]
    T, L, O, IP, RP, LP = [], [], [], [], [], []

    def pkt(t, out, ip, rport, lport, ln):
        T.append(t); O.append(out); IP.append(ip); RP.append(rport); LP.append(lport); L.append(ln)

    roots, period = [_rand_public(rng) for _ in range(rng.integers(3, 6))], rng.uniform(20, 60)
    p_root = rng.uniform(0.3, 1.0)  # root có thể không trả lời (bị chặn, mất gói)
    for t in np.arange(t0 + rng.uniform(0, period), t1, period):  # keepalive tới root
        for r in roots:
            tt = t + rng.uniform(0, 0.5)
            pkt(tt, True, r, ZT_PORT, ZT_PORT, int(rng.integers(70, 160)))
            if rng.random() < p_root:
                pkt(tt + rng.uniform(0.02, 0.3), False, r, ZT_PORT, ZT_PORT, int(rng.integers(90, 900)))
    peers = []  # mỗi peer: danh sách địa chỉ ứng viên (IP, port)
    for _ in range(rng.integers(2, 61)):
        eps = []
        for _ in range(rng.integers(1, 7)):  # IP ứng viên: mỗi IP là một node nghe 9993 và/hoặc port phụ / port NAT riêng
            ip = _rand_public(rng) if rng.random() < 0.4 else _rand_private(rng)
            ports = ([ZT_PORT] if rng.random() < 0.6 else []) + [int(p) for p in rng.integers(1024, 65536, rng.integers(0, 3))]
            eps += [(ip, p) for p in (ports or [int(rng.integers(1024, 65536))])]
        peers.append(eps)
    for _ in range(max(1, rng.poisson((t1 - t0) / rng.uniform(5, 60)))):  # dò đường
        t, p_ok, gap = rng.uniform(t0, t1), rng.uniform(0, 0.2), 10 ** rng.uniform(-5, -2)
        p_hello = 0.0 if rng.random() < 0.8 else 0.1  # đa số đợt chỉ có gói dò 70 B, một số đợt xen HELLO 145–210 B
        for i in rng.choice(len(peers), int(rng.integers(1, len(peers) + 1)), replace=False):
            for ip, port in peers[i]:
                for lp in local:
                    for _ in range(int(rng.integers(1, 3))):
                        t += gap * rng.uniform(0.5, 1.5)
                        pkt(t, True, ip, port, lp, 70 if rng.random() >= p_hello else int(rng.integers(145, 210)))
                    if rng.random() < p_ok:
                        pkt(t + rng.uniform(0.005, 0.2), False, ip, port, lp, 79)
    if rng.random() < 0.3:  # phiên dữ liệu qua ZeroTier
        ip, port = peers[rng.integers(len(peers))][0]
        lp = local[rng.integers(len(local))]
        s0 = rng.uniform(t0, t1)
        s1, rate = min(t1, s0 + rng.uniform(10, 120)), 10 ** rng.uniform(0.5, 1.5)
        for out in (True, False):
            for tt in rng.uniform(s0, s1, int((s1 - s0) * rate * rng.uniform(0.5, 1.5))):
                pkt(tt, out, ip, port, lp, int(rng.integers(90, 1400)))
    O, IP, RP, LP = np.array(O), np.array(IP, dtype=object), np.array(RP), np.array(LP)
    i32 = lambda v: pd.array(v, dtype='Int32')
    na = i32([pd.NA] * len(T))
    df = pd.DataFrame({'ts': np.array(T, float), 'len': i32(L),
                       'eth_src': np.where(O, HOST_MAC, GW_MAC), 'eth_dst': np.where(O, GW_MAC, HOST_MAC),
                       'ip_src': np.where(O, host, IP), 'ip_dst': np.where(O, IP, host),
                       'ip_proto': i32(np.full(len(T), 17)), 'ttl': i32(np.where(O, 64, 52)),
                       'tcp_sport': na, 'tcp_dport': na, 'tcp_flags': na, 'tcp_len': na,
                       'udp_sport': i32(np.where(O, LP, RP)), 'udp_dport': i32(np.where(O, RP, LP)),
                       'icmp_type': na, 'arp_op': na, 'arp_sha': '', 'arp_spa': '', 'arp_tpa': '', 'label': ''})
    return df[pf.COLS + ['label']].sort_values('ts', kind='stable', ignore_index=True)


def zt_sim_sample(job):
    rng = np.random.default_rng(job['seed'])
    bg = None
    if job['bg']:  # nền: máy chính của một file QoS cùng split, cắt 60–180 s
        bg = pd.read_parquet(job['bg'])
        dur = rng.uniform(60, 180)
        t0 = rng.uniform(bg['ts'].min(), max(bg['ts'].min(), bg['ts'].max() - dur))
        t1, host = t0 + dur, job['host']
        bg = bg[(bg['ts'] >= t0) & (bg['ts'] < t1)]
    else:  # máy chỉ chạy ZeroTier
        host = f'192.168.1.{rng.integers(2, 254)}'
        t0 = 1.75e9 + rng.uniform(0, 3e7)
        t1 = t0 + rng.uniform(120, 600)
    z = zt_sim(host, t0, t1, rng)
    pk = z if bg is None else pd.concat([bg, z], ignore_index=True).sort_values('ts', kind='stable', ignore_index=True)
    out = ZT_OUT / f"{job['name']}.parquet"
    pk.to_parquet(out, index=False)
    return {'source': 'benign_zt', 'name': job['name'], 'sample_id': f"zt_{job['name']}", 'split': job['split'],
            'scenario': 'zerotier_sim', 'file': job['bg_name'] or '', 'aug': 0, 'status': 'sim', 'zt_pkts': len(z),
            **part_row(pk, out)}


sim_jobs = []
for split, n in ZT_SIM.items():
    pool = ok[ok['split'] == split]
    for k in range(n):
        seed = zlib.crc32(f'zt_sim/{split}/{k}/{SEED}'.encode())
        r = np.random.default_rng(seed)
        b = pool.iloc[r.integers(len(pool))] if len(pool) and r.random() < ZT_SIM_BG else None
        sim_jobs.append({'name': f'sim_{split}_{k:03d}', 'split': split, 'seed': seed + 1,
                         'bg': None if b is None else b['parquet'], 'bg_name': None if b is None else b['name'],
                         'host': None if b is None else b['dev_ip']})
with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
    zsim = pd.DataFrame(list(ex.map(zt_sim_sample, sim_jobs)))
zsim.groupby('split').agg(samples=('name', 'size'), with_bg=('file', lambda s: (s != '').sum()),
                          zt_pkts=('zt_pkts', 'sum'), hours=('hours', 'sum')).round(2)

,samples,with_bg,zt_pkts,hours
split,,,,
train,300,125,1370607,20.39
val,60,22,324008,4.19


In [7]:
parts = pd.concat([ok, zt[zt['status'] != 'hong'] if len(zt) else zt, zsim], ignore_index=True)
parts.to_csv(REPORT_DIR / '01d_extra_parts.csv', index=False)
qos.to_csv(REPORT_DIR / '01d_qos_files.csv', index=False)  # cả file hỏng, để tra lại
parts.groupby(['source', 'split']).agg(parts=('name', 'size'), pkts=('pkts', 'sum'), hours=('hours', 'sum')).round(2)

parts      pkts  hours
source     split                        
benign_qos test      71   3871759   3.01
           train    223  11717146   8.78
           val       73   3426880   2.82
benign_zt  test       1       579   0.00
           train    300   6946606  20.39
           val       60   1176174   4.19

## 4. Kiểm tra nhanh bằng cửa sổ (giống 03b: mọi host LAN)

- **QoS test:** tỉ lệ gói nhận về trên gói gửi trong cửa sổ (`f_w_rx_ratio`). Trước khi gỡ NAT, tỉ lệ này gần như bằng 0 ở mọi nhóm.
- **ZeroTier:** so bản gốc với 5 bản tăng cường của file test đầu tiên. Các bản này chỉ để xem, không lưu.

In [8]:
def windows_of(path):
    try:
        return pf.window_features(pd.read_parquet(path), lan_only=True)
    except AssertionError:  # không có gói nào do host LAN gửi
        return None


qt = parts[(parts['source'] == 'benign_qos') & (parts['split'] == 'test')]
with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
    ws = list(ex.map(windows_of, qt['parquet']))
qw = pd.concat([w.assign(cat=c) for w, c in zip(ws, qt['cat']) if w is not None], ignore_index=True)
qw.groupby('cat').agg(windows=('host', 'size'), rx_ratio_median=('f_w_rx_ratio', 'median'),
                      no_rx_share=('f_w_rx_ratio', lambda s: (s == 0).mean()),
                      pps_median=('f_w_pps', 'median')).round(3)

,windows,rx_ratio_median,no_rx_share,pps_median
cat,,,,
background,243,1.76,0.000,228.750
download,5277,1.16,0.001,28264.556
game,4991,0.98,0.075,61.992
longvideo,7149,1.24,0.048,20964.361
shortvideo,7204,1.24,0.076,15984.661
videocall,5314,0.52,0.039,192.637
voicecall,2165,0.90,0.147,52.226


In [9]:
demo = zt[(zt['split'] == 'test') & (zt['aug'] == 0)] if len(zt) else zt
rows = []
if len(demo):
    pk0 = pd.read_parquet(demo['parquet'].iat[0])
    for k in range(6):
        pk, desc = (pk0, {}) if k == 0 else augment_zt(pk0, np.random.default_rng(1000 + k))
        w = pf.window_features(pk, lan_only=True)
        z = w[w['f_w_r_udp'] > 0.5]  # cửa sổ chủ yếu là UDP (ZeroTier)
        rows.append({'bản': 'gốc' if k == 0 else f'aug{k}', **desc, 'cửa sổ UDP': len(z),
                     **z[['f_w_pps', 'f_w_d_dst', 'f_w_d_port', 'f_w_rx_ratio', 'f_d_dst_60']].median().round(2)})
print('file:', demo['file'].iat[0] if len(demo) else '(chưa có pcap ZeroTier)')
pd.DataFrame(rows)

file: test/pc71.pcap


,bản,cửa sổ UDP,f_w_pps,f_w_d_dst,f_w_d_port,f_w_rx_ratio,f_d_dst_60,drop_port,resp_keep,peers,time_scale
0,gốc,13,58.43,7.0,8.0,0.06,38.0,NaN,NaN,NaN,NaN
1,aug1,20,2402.27,6.5,11.0,0.00,55.0,False,1.00,45->78,1.36
2,aug2,16,1941.38,6.5,9.5,0.00,42.0,True,0.02,45->55,0.49
3,aug3,19,706.27,7.0,11.0,0.00,54.0,False,0.00,45->72,2.09
4,aug4,17,1194.06,7.0,10.0,0.00,50.0,False,0.69,45->69,1.58
5,aug5,6,14.69,8.0,9.5,0.05,25.5,True,0.00,45->25,0.69


In [10]:
# ZeroTier mô phỏng so với pcap thật: phân vị đặc trưng của các cửa sổ chủ yếu là UDP, trên 30 mẫu train đầu tiên
def zt_windows(path):
    w = pf.window_features(pd.read_parquet(path), lan_only=True)
    return w[w['f_w_r_udp'] > 0.5]


cols = ['f_w_pps', 'f_w_d_dst', 'f_w_d_port', 'f_w_rx_ratio', 'f_w_len_mean', 'f_d_dst_60', 'f_port_per_dst_60']
sim_w = pd.concat([zt_windows(p) for p in zsim.loc[zsim['split'] == 'train', 'parquet'].head(30)], ignore_index=True)
real_w = zt_windows(demo['parquet'].iat[0]) if len(demo) else sim_w.iloc[:0]
pd.concat({'mô phỏng': sim_w[cols].quantile([0.1, 0.5, 0.9]), 'pc71 (thật)': real_w[cols].quantile([0.1, 0.5, 0.9])}).round(2)

f_w_pps  f_w_d_dst  f_w_d_port  f_w_rx_ratio  f_w_len_mean  \
mô phỏng    0.1     40.28        1.0         1.0          0.00         70.00   
            0.5   2785.75        4.0         2.0          0.32         78.62   
            0.9  41380.46        9.0        11.0          1.58        655.12   
pc71 (thật) 0.1      2.75        1.8         3.6          0.00         69.62   
            0.5     58.43        7.0         8.0          0.06         70.00   
            0.9   8964.07       10.0        11.0          0.57        130.48   

                 f_d_dst_60  f_port_per_dst_60  
mô phỏng    0.1        19.0               0.35  
            0.5       111.0               1.10  
            0.9       182.0               1.36  
pc71 (thật) 0.1        11.6               1.09  
            0.5        38.0               1.19  
            0.9        54.6               1.37